# FoundML3 — Week 3 Coding Assignment  
## Model evaluation, experimental design, and reproducibility (MedMNIST edition)

**Theme:** trustworthy evaluation for stochastic ML pipelines  
**Core tools:** pipelines • nested CV • variance reporting • imbalanced metrics

### Learning goals
- Prevent **data leakage** by putting all data-dependent steps inside a **Pipeline** and inside the resampling loop.
- Separate **model selection** (inner CV / GridSearchCV) from **model assessment** (outer CV) via **nested CV**.
- Quantify **stochasticity** by repeating the full evaluation across seeds and reporting **mean ± std**.
- Use metrics appropriate for **imbalanced** classification: ROC AUC vs PR AUC.
- Write a short **reproducibility report** describing protocol, randomization sources, leakage prevention, and reporting.

### Dataset
We use **BreastMNIST** from **MedMNIST v2**: small 28×28 grayscale ultrasound images for a (benign vs malignant) classification task.

> Note: This notebook will download the dataset the first time it runs (internet required). 


## 0. Setup

Run the next cell to install dependencies and import libraries.

In [ ]:
import sys
!{sys.executable} -m pip install medmnist

import numpy as np
import pandas as pd

GLOBAL_SEED = 42
rng = np.random.default_rng(GLOBAL_SEED)

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

import sklearn

def summarize(y, name):
    n = len(y)
    n_pos = int(y.sum())
    print(f"{name}: n={n}, positives={n_pos} ({n_pos/n:.2f}), negatives={n-n_pos} ({(n-n_pos)/n:.2f})")



## 1. Load BreastMNIST (cancer-related images) and create an imbalanced task

BreastMNIST is a **binary** classification dataset (benign vs malignant). We will treat **malignant** as the positive class.

To make the evaluation discussion more realistic, we will optionally enforce **class imbalance** in the *development pool* by subsampling positives to ~10%. This makes PR AUC (average precision) more meaningful.

### Deliverables
1. Load the dataset and build a *development pool* `X_dev, y_dev` by combining MedMNIST `train` and `val` splits.
2. Confirm which label corresponds to malignant and define `y=1` for malignant.
3. Create an imbalanced subset (target positive rate ~10%) **without changing the test set**.
4. Print class counts and the positive rate.


In [ ]:
import medmnist
from medmnist import INFO

# --- Load BreastMNIST ---
info = INFO["breastmnist"]
display(info)
DataClass = getattr(medmnist, info["python_class"])

train = DataClass(split="train", download=True)
val   = DataClass(split="val", download=True)
test  = DataClass(split="test", download=True)

# Images are 28x28 grayscale. We'll flatten to feature vectors for classical ML models.

def flatten_imgs(ds):
    X = ds.imgs.astype(np.float32) / 255.0
    X = X.reshape(len(X), -1)
    y = ds.labels.squeeze().astype(int)
    return X, y

X_tr, y_tr_raw = flatten_imgs(train)
X_va, y_va_raw = flatten_imgs(val)
X_te, y_test_raw = flatten_imgs(test)

X_dev = np.vstack([X_tr, X_va])
y_dev_raw = np.concatenate([y_tr_raw, y_va_raw])

# --- Label semantics ---
print("Label mapping:", info["label"])

# Identify which label is "malignant" and make it the positive class (y=1)
malignant_id = None
for k, v in info["label"].items():
    if isinstance(v, str) and "malig" in v.lower():
        malignant_id = int(k)
        break
if malignant_id is None:
    # fallback assumption if mapping is unexpected
    malignant_id = 1

# Make y=1 for malignant, 0 otherwise
y_dev = (y_dev_raw == malignant_id).astype(int)
y_test = (y_test_raw == malignant_id).astype(int)

summarize(y_dev, "Dev pool (train+val)")
summarize(y_test, "Test (kept unchanged)")


# Enforce imbalance in dev pool by subsampling positives to target_pos_rate
target_pos_rate = 0.10

pos_idx = np.where(y_dev == 1)[0]
neg_idx = np.where(y_dev == 0)[0]

desired_pos = int(round(target_pos_rate * len(neg_idx) / (1 - target_pos_rate)))
desired_pos = min(desired_pos, len(pos_idx))

rng = np.random.default_rng(GLOBAL_SEED)
pos_keep = rng.choice(pos_idx, size=desired_pos, replace=False)

keep_idx = np.concatenate([neg_idx, pos_keep])
rng.shuffle(keep_idx)

X_dev_imb = X_dev[keep_idx]
y_dev_imb = y_dev[keep_idx]

print("\nAfter enforcing imbalance in dev pool:")
summarize(y_dev_imb, "Dev (imbalanced)")


## 2. Metrics (threshold-free) for imbalanced classification

We will use two threshold-free metrics:
- **ROC AUC**: ranking quality (TPR vs FPR across thresholds)
- **PR AUC** (Average Precision): emphasizes positive class performance and is often more informative when positives are rare

### Deliverables
- Implement `evaluate_scores(y_true, y_score)`.


**TODO (text):** Why is PR AUC often more informative than ROC AUC when positives are rare?

In [ ]:
def evaluate_scores(y_true, y_score):
    """Return (roc_auc, pr_auc)."""
    # TODO
    raise NotImplementedError


## 3. Helper: nested CV evaluation for a pipeline + grid

Protocol:
- Outer CV: 5-fold stratified, shuffled
- Inner CV: 5-fold stratified, shuffled
- Inner scoring: `average_precision` (PR AUC)

### Deliverable
Implement `nested_cv_scores(...)` returning lists of outer-fold ROC AUC and PR AUC.
Then you can reuse it for multiple model families.


In [ ]:
def nested_cv_scores(X, y, estimator, param_grid, seed, n_splits_outer=5, n_splits_inner=5):
    """Run nested CV and return (outer_roc_list, outer_pr_list)."""
    # TODO: define outer_cv and inner_cv using StratifiedKFold(..., shuffle=True, random_state=seed)
    # TODO: for each outer fold: fit GridSearchCV on outer-train, evaluate best estimator on outer-test
    # TODO: collect roc_auc and pr_auc on the outer-test fold
    raise NotImplementedError


## 4. Two model families to compare (same protocol)

You will compare two model families under identical evaluation rules:

### A) Logistic Regression (with scaling)
Pipeline: `StandardScaler -> LogisticRegression`  
Tune: `C`: [0.01, 0.1, 1, 10, 100]

### B) Random Forest (ensemble)
Tune: `n_estimators`: [30, 60], `max_depth`: [3, 5, 10], `max_features`: ["sqrt", None]

### Deliverable
Define the two estimators and their small parameter grids.


In [ ]:
# Use the imbalanced dev pool created above:
# X_used, y_used = X_dev_imb, y_dev_imb

# TODO: A) Logistic Regression pipeline
lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=GLOBAL_SEED, n_jobs=-1))
])
lr_grid = {
    # TODO: "clf__C": [...]
}

# TODO: B) Random Forest
rf = RandomForestClassifier(random_state=GLOBAL_SEED, n_jobs=-1)
rf_grid = {
    # TODO: grid
}


## 5. Nested CV (single run) per model family

### Deliverables
- Run nested CV for Logistic Regression and Random Forest using the same seed.
- Report mean ± std across outer folds for ROC AUC and PR AUC.
- Briefly comment: do ROC AUC and PR AUC tell the same story?


In [ ]:
seed = GLOBAL_SEED

# TODO: Logistic Regression nested CV
# lr_roc, lr_pr = nested_cv_scores(X_used, y_used, lr, lr_grid, seed)
# print mean±std

# TODO: Random Forest nested CV
# rf_roc, rf_pr = nested_cv_scores(X_used, y_used, rf, rf_grid, seed)
# print mean±std


## 6. Repeated nested evaluation (variance across seeds)

A single nested-CV run is one random experiment. We repeat across seeds to quantify variance.

### Deliverables
- Use `SEEDS = [0, 1, 2, 3, 4]`.
- For each seed, run nested CV for both models and store the mean ROC AUC and mean PR AUC (across outer folds).
- Report mean ± std across seeds.
- Conclude whether the model differences are larger than the across-seed variability.


In [ ]:
SEEDS = [0, 1, 2, 3, 4]
results = []

for seed in SEEDS:
    # TODO: optionally re-instantiate models with seed for fairness
    # TODO: run nested_cv_scores for LR and RF, compute means, append to results
    pass

# TODO: summarize in a DataFrame (mean±std per model)


## 7. Optional: final hold-out test evaluation (deployment-style)

Nested CV gives an unbiased estimate of generalization. In practice, you may also keep a final **hold-out test set**.

### Deliverables
- Fit the chosen LR model family on the full dev pool (X_used, y_used) using GridSearchCV using scoring="average_precision".
- Evaluate once on the untouched MedMNIST test split (`X_te`, `y_test`) and report ROC AUC and PR AUC.


In [ ]:
# Final test evaluation (use once)
# TODO: perform one final train-on-dev / test-on-holdout evaluation


## 8. Reproducibility report (8–12 lines)

Include:
1. Protocol (outer/inner CV, seeds, metrics)
2. Randomization sources (splits, model stochasticity)
3. Leakage prevention (what is inside the pipeline / CV loop)
4. Reporting (mean±std at which level)
5. Metric rationale (ROC vs PR under imbalance)

Paste your report below.


**TODO (optional):** Write your reproducibility report here.